# DroneFuse - Step 2: per-window sensor availability and frozen features (GPU)

For every 1 s window of `splits/windows.csv` in the attached recordings:
- **availability**: how much each sensor actually delivered (camera frames, LiDAR/radar points before and after
  removing the static background, radar tracks, audio frames per node) - MMAUD's *natural* missing sensors;
- **CPU descriptors**: fixed-length LiDAR/radar descriptors (all points and foreground), point sets for Step 3,
  audio log-mel spectrograms of all 4 nodes;
- **GPU embeddings** (frozen, nothing trained): ConvNeXt-T on the native camera frames (average and max pooled),
  AST (AudioSet) on each audio node;
- a **check on the validation split only**: one linear classifier per sensor (closed protocol).

**Kaggle settings:** Accelerator = **GPU T4 x2** (or P100). Internet = **On** (downloads the two pretrained models).
**Inputs to add:** `dronefuse-code` and whichever of `dronefuse-mmaud-v1`, `dronefuse-mmaud-v2a`,
`dronefuse-mmaud-v2b` exist. Missing groups are skipped - run again when they exist.
**Safe for "Save & Run All":** each part is one step; an error is recorded and the next step runs;
`step2_report.txt` is always written. **Expected run time:** about 1-2 GPU-hours for all 12 recordings.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
from dronefuse.features import FeatureSettings

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step2')
ONLY_RECORDINGS = None            # e.g. ['V1_Mavic3'] to process one recording
TIME_BUDGET_MINUTES = 11 * 60
CPU_WORKERS = os.cpu_count() or 4
SETTINGS = FeatureSettings()      # voxel 0.5 m (LiDAR) / 1.0 m (radar); background = occupied in >= 10 % of scans

In [ ]:
import json

from dronefuse import environment
from dronefuse.runner import StepRunner
from dronefuse.step2 import find_recordings
from dronefuse.windows import load_windows

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'step2_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
windows_csv = code_root / 'splits' / 'windows.csv'
if not windows_csv.exists():
    windows_csv = next(INPUT.rglob('windows.csv'))
windows = load_windows(windows_csv)
recordings = find_recordings(INPUT)
if ONLY_RECORDINGS:
    recordings = {k: v for k, v in recordings.items() if k in ONLY_RECORDINGS}
print(len(windows), 'windows in', windows_csv)
print('recordings found:', {k: str(v) for k, v in recordings.items()})
missing = sorted({w['recording'] for w in windows} - set(recordings))
print('not attached (skipped):', missing)
summary['recordings_found'] = {k: str(v) for k, v in recordings.items()}
save_summary()

In [ ]:
# CPU part: one process per recording (a few minutes per recording).
from dronefuse.step2 import cpu_features

cpu_results = runner.run('CPU features', cpu_features, recordings, windows, OUTPUT, SETTINGS, CPU_WORKERS) or {}
summary['cpu'] = cpu_results
save_summary()

In [ ]:
# GPU part: frozen camera and audio embeddings, one step per recording.
from dronefuse.step2 import deep_features

deep_results = {}
for name, folder in recordings.items():
    if 'error' in cpu_results.get(name, {'error': 'no CPU features'}):
        continue
    rows = [w for w in windows if w['recording'] == name]
    deep_results[name] = runner.run(f'GPU embeddings {name}', deep_features, name, folder, rows, OUTPUT)
    summary['deep'] = deep_results
    save_summary()

In [ ]:
# Check on the validation split only: one linear classifier per sensor, closed protocol.
from dronefuse.step2 import format_step2_report, linear_probe

done = [n for n, r in cpu_results.items() if 'error' not in r]
probe = runner.run('linear probe (val only)', linear_probe, OUTPUT, windows, done)
summary['probe'] = probe

report = format_step2_report(cpu_results, deep_results, probe, summary.get('environment'))
failed = runner.failed()
report += '\n\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed))
report += '\nRECORDINGS NOT ATTACHED: ' + (', '.join(missing) or 'none')
saved = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
report += f'\nSAVED: {saved / 1e9:.2f} GB in {OUTPUT}'
(OUTPUT / 'step2_report.txt').write_text(report)
save_summary()
print(report)

## After the run

1. Send back `dronefuse_outputs/step2/step2_report.txt` (or the printed report above).
2. When all 12 recordings were processed: open this notebook's saved version -> **Output** -> **New Dataset**,
   name it **`dronefuse-step2-features`** (private). Step 3 (fusion, missing sensors, unknown drones) reads it.